# LBO-style sources & uses

High-level **leveraged buyout** mechanics: sources/uses, entry multiple, layered debt, amortization, and a simple **equity IRR / MOIC** grid.

## Concept

We approximate an LBO with explicit **debt balances**, **mandatory amortization**, and an **operating forecast**. Exit equity is distributable value after repaying remaining debt. Sensitivity on **entry vs exit EV/EBITDA** is a two-way table built by re-evaluating variants.

The transaction itself is **not** hand-assembled in Python. `statements_analytics.evaluate_lbo` prices the entry enterprise value off a model node, balances sources against uses, solves the sponsor equity check as the residual, and returns exit proceeds and MOIC. Tranche balances come from the Rust roll-forward template `add_roll_forward_with_opening` (opening balance = funded amount at close, mandatory amortization as the decrease) rather than a hand-typed balance series.

**IRR is deliberately out of scope for `evaluate_lbo`.** Pair its `equity_check` and `exit_equity_proceeds` with `portfolio.mwr_xirr`, which already owns dated-cash-flow XIRR.

**Static model caveat:** Amortization is a fixed mandatory schedule — it doesn't dynamically adjust from cash flow. Interest is computed from tranche balances but is not subtracted from UFCF. A full LBO model would compute levered free cash flow (`lfcf = ufcf - total_cash_interest`) and derive debt paydown from available cash (mandatory amortization + optional sweep). This demo focuses on the **entry/exit economics** and sensitivity grid rather than the full cash sweep waterfall.

In [1]:
import json

import sys
sys.path.insert(0, "../..")

from _shared import series
from finstack_quant.portfolio import mwr_xirr
from finstack_quant.statements import Evaluator, FinancialModelSpec, ModelBuilder
from finstack_quant.statements_analytics import (
    SensitivityConfig,
    add_roll_forward_with_opening,
    evaluate_lbo,
    goal_seek,
    run_sensitivity,
)

PERIODS = ["2025", "2026", "2027", "2028", "2029"]
EXIT_PERIOD = "2029"

BASE_ENTRY_MULTIPLE = 8.5
BASE_EXIT_MULTIPLE = 9.5
TRANSACTION_FEES = 3.0

# (tranche, funded amount at close, cash coupon, annual straight-line amortization
#  as a fraction of the funded amount). Mezzanine is bullet, so it amortizes 0%.
BASE_TRANCHES = [
    ("term_loan_a", 90.0, 0.06, 0.20),
    ("revolver", 10.0, 0.08, 0.20),
    ("mezzanine", 15.0, 0.12, 0.00),
]

def build_lbo_model(entry_multiple, exit_multiple):
    """Operating model + capital structure for one (entry, exit) multiple pair.

    Debt is sized off the entry multiple, and each tranche balance is produced by
    the Rust roll-forward template (``add_roll_forward_with_opening``) rather than
    a hand-typed balance series: opening balance = funded amount at close, and the
    mandatory amortization node is the only decrease.
    """
    debt_scale = entry_multiple / BASE_ENTRY_MULTIPLE
    tranches = [(name, funded * debt_scale, rate, amort) for name, funded, rate, amort in BASE_TRANCHES]

    b = ModelBuilder("lbo-demo")
    b.periods("2025..2029", "2025")
    b.with_meta("currency", json.dumps("USD"))

    # Operating model
    b.value("revenue", series(PERIODS, [100.0, 105.0, 110.0, 115.0, 120.0]))
    b.compute("ebitda", "revenue * 0.22")
    b.value("capex", series(PERIODS, [5.0, 5.0, 5.5, 5.5, 6.0]))
    b.value("delta_nwc", series(PERIODS, [1.0, 1.0, 1.0, 1.0, 1.0]))
    b.compute("ufcf", "ebitda - capex - delta_nwc")

    # Capital structure. `<name>_end` is created below by the roll-forward template;
    # forward references from compute nodes are resolved at evaluation time.
    for name, funded, rate, amort_pct in tranches:
        b.value(f"{name}_amort", series(PERIODS, [0.0] + [funded * amort_pct] * 4))
        b.value(f"{name}_rate", series(PERIODS, [rate] * 5))
        b.compute(f"{name}_interest", f"{name}_end * {name}_rate")
    b.compute("total_debt", " + ".join(f"{name}_end" for name, _, _, _ in tranches))
    b.compute("total_cash_interest", " + ".join(f"{name}_interest" for name, _, _, _ in tranches))

    # In-model exit proceeds node: this is the hook `goal_seek` and `run_sensitivity`
    # shock, since those tools operate on statement NODES, not on `evaluate_lbo` args.
    b.value("exit_multiple_scalar", series(PERIODS, [0.0, 0.0, 0.0, 0.0, exit_multiple]))
    b.compute("equity_proceeds", "ebitda * exit_multiple_scalar - total_debt")

    model_json = b.build().to_json()
    for name, funded, _, _ in tranches:
        model_json = add_roll_forward_with_opening(model_json, name, [], [f"{name}_amort"], funded)
    return FinancialModelSpec.from_json(model_json), tranches

spec, tranches = build_lbo_model(BASE_ENTRY_MULTIPLE, BASE_EXIT_MULTIPLE)
res = Evaluator().evaluate(spec)
model_json = spec.to_json()

# Transaction economics come from Rust: entry EV, sources & uses, the residual
# sponsor equity check, exit EV/proceeds and MOIC.
lbo = evaluate_lbo(
    spec,
    entry_multiple=BASE_ENTRY_MULTIPLE,
    entry_metric_node="ebitda",
    exit_multiple=BASE_EXIT_MULTIPLE,
    exit_metric_node="ebitda",
    exit_net_debt_node="total_debt",
    exit_period=EXIT_PERIOD,
    sources=[(name, funded) for name, funded, _, _ in tranches],
    transaction_fees=TRANSACTION_FEES,
)

ccy = lbo["currency"]
print(f"Entry EBITDA: {lbo['entry_metric']} | Entry EV: {lbo['entry_enterprise_value']} {ccy}")
print(f"Sources: {lbo['sources_total']} (debt {lbo['debt_total']} + equity {lbo['equity_check']})")
print(f"Uses:    {lbo['uses_total']} (EV {lbo['entry_enterprise_value']} + fees {TRANSACTION_FEES})")
print("Sources & uses balanced:", lbo["sources_uses_balanced"])
print(f"Exit EBITDA: {lbo['exit_metric']} | Exit EV: {lbo['exit_enterprise_value']} | Net debt: {lbo['exit_net_debt']}")
print(f"Exit equity proceeds: {lbo['exit_equity_proceeds']} | MOIC: {round(lbo['moic'], 3)}x")

# Cross-check: the in-model `equity_proceeds` node must agree with the Rust
# transaction result. They are two routes to the same number, so a mismatch means
# the statement model and the LboConfig have drifted apart.
in_model_ep = res.get("equity_proceeds", EXIT_PERIOD)
print("In-model equity_proceeds node:", round(in_model_ep, 6), "| agrees:", abs(in_model_ep - lbo["exit_equity_proceeds"]) < 1e-9)

# Tranche balances now come from the roll-forward template.
print("--- tranche closing balances (`<name>_end`) ---")
for name, _, _, _ in tranches:
    print(f"  {name:12s}", [round(res.get(f'{name}_end', p), 2) for p in PERIODS])
print("  total_debt   ", [round(res.get("total_debt", p), 2) for p in PERIODS])
print("  cash interest", [round(res.get("total_cash_interest", p), 3) for p in PERIODS])

# Rust-owned XIRR on sponsor cash flows. IRR is deliberately out of scope for
# `evaluate_lbo`; pair its equity check and exit proceeds with `mwr_xirr`.
equity_cashflows = json.dumps([
    {"date": "2025-01-01", "amount": -lbo["equity_check"]},
    {"date": "2030-01-01", "amount": lbo["exit_equity_proceeds"]},
])
print("Equity IRR (annual XIRR):", round(mwr_xirr(equity_cashflows), 4))

sens_cfg = SensitivityConfig(
    mode="Diagonal",
    parameters=[("revenue", "2029", 120.0, [108.0, 120.0, 132.0])],
    target_metrics=["equity_proceeds", "ufcf"],
)
sens_result = run_sensitivity(spec, sens_cfg)
print("--- run_sensitivity (revenue@2029 vs targets) ---")
print("Scenarios:", len(sens_result), "targets:", sens_result.target_metrics)


Entry EBITDA: 22.0 | Entry EV: 187.0 USD
Sources: 190.0 (debt 115.0 + equity 75.0)
Uses:    190.0 (EV 187.0 + fees 3.0)
Sources & uses balanced: True
Exit EBITDA: 26.4 | Exit EV: 250.8 | Net debt: 35.0
Exit equity proceeds: 215.8 | MOIC: 2.877x
In-model equity_proceeds node: 215.8 | agrees: True
--- tranche closing balances (`<name>_end`) ---
  term_loan_a  [90.0, 72.0, 54.0, 36.0, 18.0]
  revolver     [10.0, 8.0, 6.0, 4.0, 2.0]
  mezzanine    [15.0, 15.0, 15.0, 15.0, 15.0]
  total_debt    [115.0, 95.0, 75.0, 55.0, 35.0]
  cash interest [8.0, 6.76, 5.52, 4.28, 3.04]
Equity IRR (annual XIRR): 0.2352
--- run_sensitivity (revenue@2029 vs targets) ---
Scenarios: 3 targets: ['equity_proceeds', 'ufcf']


In [2]:
print("Entry multiple x Exit multiple -> MOIC")
print("(debt is sized off the entry multiple, so the sponsor equity check is")
print(" re-solved as the sources-and-uses residual at every entry multiple)")

for em in (7.5, 8.5, 9.5):
    row, equity_check = [], None
    for xm in (8.0, 9.5, 11.0):
        grid_spec, grid_tranches = build_lbo_model(em, xm)
        grid_lbo = evaluate_lbo(
            grid_spec,
            entry_multiple=em,
            entry_metric_node="ebitda",
            exit_multiple=xm,
            exit_metric_node="ebitda",
            exit_net_debt_node="total_debt",
            exit_period=EXIT_PERIOD,
            sources=[(name, funded) for name, funded, _, _ in grid_tranches],
            transaction_fees=TRANSACTION_FEES,
        )
        row.append(round(grid_lbo["moic"], 3))
        equity_check = grid_lbo["equity_check"]  # independent of the exit multiple
    print(f"  {em}x entry (equity check {round(equity_check, 1)}):", row)

solved, _ = goal_seek(
    model_json,
    target_node="equity_proceeds",
    target_period="2029",
    target_value=120.0,
    driver_node="revenue",
    driver_period="2029",
    update_model=False,
)
print("goal_seek revenue@2029 for equity_proceeds=120:", solved)


Entry multiple x Exit multiple -> MOIC
(debt is sized off the entry multiple, so the sponsor equity check is
 re-solved as the sources-and-uses residual at every entry multiple)
  7.5x entry (equity check 66.5): [2.71, 3.306, 3.901]
  8.5x entry (equity check 75.0): [2.349, 2.877, 3.405]
  9.5x entry (equity check 83.5): [2.062, 2.536, 3.01]
goal_seek revenue@2029 for equity_proceeds=120: 74.16267942583733


## Takeaways

- **`evaluate_lbo` owns the transaction:** entry EV from a model node, sources & uses closure, the residual sponsor equity check, exit EV/net debt/proceeds, and MOIC — no hand-rolled deal arithmetic in the notebook.
- **`add_roll_forward_with_opening` owns tranche balances:** funded amount as the opening balance, mandatory amortization as the decrease. `<name>_beg` / `<name>_end` nodes drop straight into interest and `total_debt` formulas.
- **Tranche-level interest** feeds cash needs; principal schedules drive **exit total debt** and **sponsor proceeds**.
- **MOIC comes from `evaluate_lbo`; IRR comes from `portfolio.mwr_xirr`** — the two live in different crates on purpose.
- `goal_seek` and `run_sensitivity` shock statement **nodes**, so the in-model `equity_proceeds` node stays as their hook and is cross-checked against the `evaluate_lbo` result.